# Advisory surface replay demo

Задача: загрузить обученные `rotation_model` и `speed_model`, выбрать точку из replay-скважины и построить интерактивную Plotly-поверхность:

```text
pressure_axis × pressure_rotation → predicted_future_speed
```

На графике показываются:
- поверхность текущей энергоёмкости;
- текущая точка оператора;
- recommended optimum;
- линия current → recommended.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import joblib

RANDOM_STATE = 42
EPS = 1e-6
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load data and artifacts

In [2]:
DATA_PATH = "united_rock_energy_segment_quantile.csv"
ARTIFACT_DIR = Path("drilling_advisory_artifacts")

if not Path(DATA_PATH).exists():
    raise FileNotFoundError("Не найден united_rock_energy_segment_quantile.csv")
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError("Не найдена папка drilling_advisory_artifacts. Сначала запусти train_drilling_advisory_models.ipynb")

df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")
df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)
df["rock_energy_type_final"] = df["rock_energy_type_final"].fillna("unknown").astype(str)

rotation_model = joblib.load(ARTIFACT_DIR / "rotation_model.joblib")
speed_model = joblib.load(ARTIFACT_DIR / "speed_model.joblib")

with open(ARTIFACT_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
with open(ARTIFACT_DIR / "surface_ranges_by_energy_type.json", "r", encoding="utf-8") as f:
    surface_ranges = json.load(f)

base_numeric_features = feature_config["base_numeric_features"]
categorical_features = feature_config["categorical_features"]
speed_numeric_features = feature_config["speed_numeric_features"]

print("Loaded", df.shape)
print("Artifacts loaded")

Loaded (415049, 118)
Artifacts loaded


## 2. Recreate features

In [3]:
def add_features(df):
    df = df.copy()
    df["total_pressure"] = df["pressure_axis"] + df["pressure_rotation"]
    df["pressure_balance"] = df["pressure_axis"] / (df["pressure_axis"] + df["pressure_rotation"] + EPS)
    df["axis_over_rot_pressure"] = df["pressure_axis"] / (df["pressure_rotation"] + EPS)
    df["rot_pressure_over_axis"] = df["pressure_rotation"] / (df["pressure_axis"] + EPS)
    df["rotation_efficiency"] = df["rotation"] / (df["pressure_rotation"] + EPS)
    df["axis_x_rotation"] = df["pressure_axis"] * df["rotation"]
    df["rot_pressure_x_rotation"] = df["pressure_rotation"] * df["rotation"]
    df["energy_input_proxy"] = df["pressure_axis"] + df["pressure_rotation"] * df["rotation"]
    df["log_energy_input_proxy"] = np.log1p(df["energy_input_proxy"])
    df["dt"] = df.groupby("well_id")["processing_time"].diff().dt.total_seconds()
    df["dt"] = df["dt"].fillna(df["dt"].median())

    history_cols = ["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth", "energy_input_proxy", "pressure_balance"]
    for col in history_cols:
        for lag in [1, 3, 6, 12]:
            df[f"{col}_lag{lag}"] = df.groupby("well_id")[col].shift(lag)
        shifted = df.groupby("well_id")[col].shift(1)
        for w in [6, 12, 30, 60]:
            min_p = max(2, w // 3)
            df[f"{col}_roll_mean_{w}"] = shifted.groupby(df["well_id"]).rolling(w, min_periods=min_p).mean().reset_index(level=0, drop=True)
            df[f"{col}_roll_std_{w}"] = shifted.groupby(df["well_id"]).rolling(w, min_periods=min_p).std().reset_index(level=0, drop=True)
    for col in ["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]:
        prev = df.groupby("well_id")[col].shift(1)
        df[f"{col}_diff1"] = df[col] - prev
        df[f"{col}_rel_diff1"] = df[f"{col}_diff1"] / (prev.abs() + EPS)
    return df

df = add_features(df)
model_df = df.dropna(subset=base_numeric_features + categorical_features).copy()
print(model_df.shape)

(372399, 188)


## 3. Build recommendation surface

In [4]:
def recompute_candidate_features(grid):
    grid = grid.copy()
    grid["total_pressure"] = grid["pressure_axis"] + grid["pressure_rotation"]
    grid["pressure_balance"] = grid["pressure_axis"] / (grid["pressure_axis"] + grid["pressure_rotation"] + EPS)
    grid["axis_over_rot_pressure"] = grid["pressure_axis"] / (grid["pressure_rotation"] + EPS)
    grid["rot_pressure_over_axis"] = grid["pressure_rotation"] / (grid["pressure_axis"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["axis_x_rotation"] = grid["pressure_axis"] * grid["rotation"]
    grid["rot_pressure_x_rotation"] = grid["pressure_rotation"] * grid["rotation"]
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]
    grid["log_energy_input_proxy"] = np.log1p(grid["energy_input_proxy"])
    return grid

def build_surface(row, grid_size=60, max_delta_frac=0.08):
    et = row["rock_energy_type_final"]
    if et in surface_ranges:
        r = surface_ranges[et]
        p_ax_low, p_ax_high = r["pressure_axis_q05"], r["pressure_axis_q95"]
        p_rot_low, p_rot_high = r["pressure_rotation_q05"], r["pressure_rotation_q95"]
    else:
        p_ax_low, p_ax_high = model_df["pressure_axis"].quantile([.05,.95])
        p_rot_low, p_rot_high = model_df["pressure_rotation"].quantile([.05,.95])

    cur_ax, cur_rotp = row["pressure_axis"], row["pressure_rotation"]
    p_ax_min = max(p_ax_low, cur_ax * (1 - max_delta_frac))
    p_ax_max = min(p_ax_high, cur_ax * (1 + max_delta_frac))
    p_rot_min = max(p_rot_low, cur_rotp * (1 - max_delta_frac))
    p_rot_max = min(p_rot_high, cur_rotp * (1 + max_delta_frac))
    if p_ax_min >= p_ax_max:
        p_ax_min, p_ax_max = cur_ax * (1 - max_delta_frac), cur_ax * (1 + max_delta_frac)
    if p_rot_min >= p_rot_max:
        p_rot_min, p_rot_max = cur_rotp * (1 - max_delta_frac), cur_rotp * (1 + max_delta_frac)

    p_ax_grid = np.linspace(p_ax_min, p_ax_max, grid_size)
    p_rot_grid = np.linspace(p_rot_min, p_rot_max, grid_size)
    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)
    grid = pd.DataFrame({"pressure_axis": PA.ravel(), "pressure_rotation": PR.ravel()})

    derived = {"pressure_axis", "pressure_rotation", "total_pressure", "pressure_balance", "axis_over_rot_pressure", "rot_pressure_over_axis", "rotation_efficiency", "axis_x_rotation", "rot_pressure_x_rotation", "energy_input_proxy", "log_energy_input_proxy"}
    for col in base_numeric_features:
        if col not in derived:
            grid[col] = row[col]
    for col in categorical_features:
        grid[col] = row[col]
    grid = recompute_candidate_features(grid)

    grid["candidate_future_rotation"] = rotation_model.predict(grid[base_numeric_features + categorical_features])
    grid["pred_future_speed"] = speed_model.predict(grid[speed_numeric_features + categorical_features])

    Z = grid["pred_future_speed"].values.reshape(PA.shape)
    best_idx = int(grid["pred_future_speed"].values.argmax())
    best = grid.iloc[best_idx]

    current_grid = pd.DataFrame([row[base_numeric_features + categorical_features].to_dict()])
    current_grid["candidate_future_rotation"] = rotation_model.predict(current_grid[base_numeric_features + categorical_features])
    current_pred_speed = float(speed_model.predict(current_grid[speed_numeric_features + categorical_features])[0])

    return PA, PR, Z, grid, best, current_pred_speed

## 4. Select replay point and plot

In [5]:
# Можно выбрать конкретный well_id и индекс.
WELL_ID = model_df["well_id"].dropna().unique()[0]
POINT_INDEX_IN_WELL = 250

well_df = model_df[model_df["well_id"] == WELL_ID].copy()
row = well_df.iloc[min(POINT_INDEX_IN_WELL, len(well_df)-1)]

PA, PR, Z, grid, best, current_pred_speed = build_surface(row, grid_size=60, max_delta_frac=0.08)

current = {
    "pressure_axis": float(row["pressure_axis"]),
    "pressure_rotation": float(row["pressure_rotation"]),
    "current_speed": float(row["speed"]),
    "current_predicted_future_speed": current_pred_speed,
    "energy_type": row["rock_energy_type_final"],
}

recommended = {
    "pressure_axis": float(best["pressure_axis"]),
    "pressure_rotation": float(best["pressure_rotation"]),
    "predicted_future_speed": float(best["pred_future_speed"]),
    "predicted_future_rotation": float(best["candidate_future_rotation"]),
}

print("Current:", current)
print("Recommended:", recommended)
print("Predicted uplift %:", 100 * (recommended["predicted_future_speed"] / (current_pred_speed + EPS) - 1))

fig = go.Figure()
fig.add_trace(go.Surface(x=PA, y=PR, z=Z, colorscale="Viridis", opacity=0.85, name="future ROP surface"))
fig.add_trace(go.Scatter3d(
    x=[current["pressure_axis"]], y=[current["pressure_rotation"]], z=[current_pred_speed],
    mode="markers+text", text=["current"], textposition="top center",
    marker=dict(size=7, color="red"), name="current operator point"
))
fig.add_trace(go.Scatter3d(
    x=[recommended["pressure_axis"]], y=[recommended["pressure_rotation"]], z=[recommended["predicted_future_speed"]],
    mode="markers+text", text=["recommended"], textposition="top center",
    marker=dict(size=8, color="yellow", symbol="diamond"), name="recommended optimum"
))
fig.add_trace(go.Scatter3d(
    x=[current["pressure_axis"], recommended["pressure_axis"]],
    y=[current["pressure_rotation"], recommended["pressure_rotation"]],
    z=[current_pred_speed, recommended["predicted_future_speed"]],
    mode="lines", line=dict(width=6, color="white"), name="current → recommended"
))
fig.update_layout(
    title=f"Future ROP surface | well={WELL_ID} | energy={current['energy_type']}",
    scene=dict(xaxis_title="pressure_axis", yaxis_title="pressure_rotation", zaxis_title="predicted future speed"),
    height=800,
)
fig.show()

Current: {'pressure_axis': 20056.0, 'pressure_rotation': 15603.0, 'current_speed': 0.0181800000000002, 'current_predicted_future_speed': 0.012125057955991364, 'energy_type': 'medium_high_energy'}
Recommended: {'pressure_axis': 20572.696949152545, 'pressure_rotation': 16555.04745762712, 'predicted_future_speed': 0.01257001635185686, 'predicted_future_rotation': 100.48937214140157}
Predicted uplift %: 3.6611930891039535


## 5. Plot several points to see surface switching

In [6]:
# Эта ячейка строит поверхности для нескольких точек одной скважины.
# Удобно смотреть, как меняется energy_type и surface.
POINTS = [100, 250, 500, 750]

for idx in POINTS:
    if idx >= len(well_df):
        continue
    row = well_df.iloc[idx]
    PA, PR, Z, grid, best, current_pred_speed = build_surface(row, grid_size=45, max_delta_frac=0.08)
    uplift = 100 * (float(best["pred_future_speed"]) / (current_pred_speed + EPS) - 1)

    fig = go.Figure()
    fig.add_trace(go.Surface(x=PA, y=PR, z=Z, colorscale="Viridis", opacity=0.85))
    fig.add_trace(go.Scatter3d(
        x=[row["pressure_axis"]], y=[row["pressure_rotation"]], z=[current_pred_speed],
        mode="markers+text", text=["current"], marker=dict(size=7, color="red")
    ))
    fig.add_trace(go.Scatter3d(
        x=[best["pressure_axis"]], y=[best["pressure_rotation"]], z=[best["pred_future_speed"]],
        mode="markers+text", text=["recommended"], marker=dict(size=8, color="yellow", symbol="diamond")
    ))
    fig.update_layout(
        title=f"idx={idx} | energy={row['rock_energy_type_final']} | uplift={uplift:.2f}%",
        scene=dict(xaxis_title="pressure_axis", yaxis_title="pressure_rotation", zaxis_title="predicted future speed"),
        height=750,
    )
    fig.show()